# Paired Microscopy Cancer Classification

Binary classification of bright-field (BF) and fluorescence (FL) images with PyTorch.

## Workflow

1. Discover image folders and labels.
2. Build six-channel BF/FL tensors with shared geometric augmentation.
3. Adapt pretrained EfficientNet-B0 for binary classification.
4. Create a stratified patient split.
5. Monitor loss and AUC; schedule learning rates and save checkpoints.
6. Export predictions and learning curves.

## Runtime

Attach data under `/kaggle/input/`. Outputs use `/kaggle/working/`, or the current directory locally.


## Environment and experiment settings


In [ ]:
%pip install -q torch torchvision scikit-learn pandas numpy pillow matplotlib


In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from torchvision.transforms import functional as TF


# Keep experiment settings together so training and prediction use matching defaults.
CONFIG = {
    "seed": 42,
    "validation_fraction": 0.2,
    "batch_size": 16,
    "epochs": 10,
    "learning_rate": 1e-5,
    "weight_decay": 1e-3,
    "head_dropout": 0.4,
    "lr_factor": 0.5,
    "lr_patience": 2,
    "early_stopping_patience": 5,
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


## Find the dataset

In Kaggle, select **Add Input → Competition → Multimodal Cancer Classification Challenge 2026**.
Discovery uses `train.csv` and `sampleSubmission.csv`; a directory hint can also be supplied.


In [ ]:
def find_dataset_dir(hint=None):
    roots = ([Path(hint)] if hint is not None else []) + [
        Path("/kaggle/input/competitions/multimodal-cancer-classification-challenge-2026"),
        Path("/kaggle/input/multimodal-cancer-classification-challenge-2026"),
        Path("/kaggle/input"),
        Path("."),
    ]
    roots = list(dict.fromkeys(root.resolve() for root in roots if root.is_dir()))

    def contains_tables(folder):
        return all((folder / name).is_file() for name in ("train.csv", "sampleSubmission.csv"))

    # Check likely locations before searching their descendants.
    for root in roots:
        if contains_tables(root):
            return root

    searched = []
    for root in roots:
        if any(parent == root or parent in root.parents for parent in searched):
            continue
        for table in root.rglob("train.csv"):
            if contains_tables(table.parent):
                return table.parent
        searched.append(root)

    locations = ", ".join(str(root) for root in roots) or "(no existing directories)"
    raise FileNotFoundError(f"Expected train.csv and sampleSubmission.csv beneath: {locations}")


BASE_DIR = find_dataset_dir()
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

BF_TRAIN_DIR, FL_TRAIN_DIR = (BASE_DIR / modality / "train" for modality in ("BF", "FL"))
BF_TEST_DIR, FL_TEST_DIR = (BASE_DIR / modality / "test" for modality in ("BF", "FL"))
TRAIN_CSV = BASE_DIR / "train.csv"
SAMPLE_SUB = BASE_DIR / "sampleSubmission.csv"

print(f"Data: {BASE_DIR}\nOutputs: {WORKING_DIR.resolve()}")
for folder in (BF_TRAIN_DIR, FL_TRAIN_DIR, BF_TEST_DIR, FL_TEST_DIR):
    if not folder.is_dir():
        raise FileNotFoundError(f"Missing image folder: {folder}")
    print(f"  Ready: {folder.relative_to(BASE_DIR)}")


## Paired images and augmentation


In [ ]:
class PairedImageTransform:
    def __init__(self, training=False):
        self.training = training
        self.color_jitter = T.ColorJitter(0.2, 0.2, 0.1, 0.05)

    def __call__(self, bf, fl):
        if bf.size != fl.size:
            raise ValueError(f"Paired image sizes differ: BF={bf.size}, FL={fl.size}")

        if self.training:
            # Draw geometry once so corresponding pixels stay aligned.
            if torch.rand(()) < 0.5:
                bf, fl = TF.hflip(bf), TF.hflip(fl)
            if torch.rand(()) < 0.5:
                bf, fl = TF.vflip(bf), TF.vflip(fl)
            angle = T.RandomRotation.get_params((-15, 15))
            bf, fl = TF.rotate(bf, angle), TF.rotate(fl, angle)

            # Each modality retains its own random intensity perturbation.
            bf, fl = self.color_jitter(bf), self.color_jitter(fl)

        # Preserve input resolution and the original [0, 1] scaling in both modes.
        return torch.cat((TF.to_tensor(bf), TF.to_tensor(fl)), dim=0)


class CancerDataset(Dataset):
    def __init__(self, bf_dir, fl_dir, records=None, transform=None):
        self.bf_dir, self.fl_dir = Path(bf_dir), Path(fl_dir)
        self.transform = transform if transform is not None else PairedImageTransform()

        # Cache metadata once instead of indexing a DataFrame for every image.
        if records is not None:
            if not isinstance(records, pd.DataFrame):
                records = pd.read_csv(records)
            self.names = records["Name"].to_numpy(copy=True)
            self.labels = torch.from_numpy(records["Diagnosis"].to_numpy(dtype=np.float32, copy=True))
        else:
            self.names = sorted(path.name for path in self.bf_dir.iterdir() if path.is_file())
            self.labels = None

        if len(self.names) == 0:
            raise ValueError("The image dataset is empty.")

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        # Close both source files after decoding their RGB pixels.
        with Image.open(self.bf_dir / name) as bf_source, Image.open(self.fl_dir / name) as fl_source:
            bf, fl = bf_source.convert("RGB"), fl_source.convert("RGB")

        image = self.transform(bf, fl)
        target = name if self.labels is None else self.labels[index]
        return image, target


## Six-channel EfficientNet-B0


In [ ]:
class CancerNet(nn.Module):
    def __init__(self, weights=EfficientNet_B0_Weights.DEFAULT):
        super().__init__()
        self.model = efficientnet_b0(weights=weights)
        stem = self.model.features[0][0]

        # Expand the input stem while retaining its spatial settings.
        paired_stem = nn.Conv2d(
            6, stem.out_channels, kernel_size=stem.kernel_size,
            stride=stem.stride, padding=stem.padding, bias=False,
        )
        with torch.no_grad():
            paired_stem.weight[:, :3].copy_(stem.weight)
            # Broadcast the RGB mean into the three fluorescence channels.
            paired_stem.weight[:, 3:].copy_(stem.weight.mean(dim=1, keepdim=True).expand(-1, 3, -1, -1))
        self.model.features[0][0] = paired_stem

        # Retain the backbone dropout, followed by the binary classification head.
        feature_count = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Sequential(
            nn.Dropout(p=CONFIG["head_dropout"]),
            nn.Linear(feature_count, 1),
        )

    def forward(self, images):
        return self.model(images)


## Training and AUC evaluation


In [ ]:
def make_loader(dataset, device, shuffle=False):
    # Pinned host memory supports asynchronous copies to a CUDA device.
    return DataLoader(
        dataset, batch_size=CONFIG["batch_size"], shuffle=shuffle,
        pin_memory=device.type == "cuda",
    )


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    losses = []
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True).unsqueeze(1)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), targets)
        loss.backward()
        optimizer.step()
        losses.append(loss.detach())

    # Preserve the batch-mean loss, with one device synchronization per epoch.
    return torch.stack(losses).mean(dtype=torch.float64).item()


@torch.inference_mode()
def evaluate_auc(model, loader, device):
    model.eval()
    probabilities, targets = [], []
    for images, labels in loader:
        logits = model(images.to(device, non_blocking=True))
        probabilities.append(logits.sigmoid().flatten())
        targets.append(labels)

    return roc_auc_score(
        torch.cat(targets).cpu().numpy(),
        torch.cat(probabilities).cpu().numpy(),
    )


## Patient split and model fitting


In [ ]:
def split_patients(records):
    records = records.copy()
    if not records["Diagnosis"].isin([0, 1]).all():
        raise ValueError("Diagnosis must contain binary labels 0 and 1.")
    if records["Name"].isna().any():
        raise ValueError("Every image must have a filename.")

    # Filenames share the patient identifier before the '_image_' separator.
    records["PatientID"] = records["Name"].str.split("_image_", n=1).str[0]
    grouped = records.groupby("PatientID")["Diagnosis"]
    if grouped.nunique().ne(1).any():
        raise ValueError("Images belonging to one patient must share a diagnosis.")
    patients = grouped.first().reset_index()

    # Stratify one row per patient, then recover every image for the selected groups.
    training, validation = train_test_split(
        patients, test_size=CONFIG["validation_fraction"],
        random_state=CONFIG["seed"], stratify=patients["Diagnosis"],
    )
    train_records = records.loc[records["PatientID"].isin(training["PatientID"])].copy()
    val_records = records.loc[records["PatientID"].isin(validation["PatientID"])].copy()
    return train_records, val_records


def save_learning_curves(history):
    epochs = range(1, len(history["loss"]) + 1)
    plots = [
        ("auc_curve_p.png", "Training and Validation AUC", "AUC",
         [("Train AUC", history["train_auc"]), ("Validation AUC", history["val_auc"])]),
        ("loss_curve_p.png", "Training Loss", "Loss", [(None, history["loss"])]),
    ]
    # Share plotting setup while preserving separate AUC and loss figures.
    for filename, title, ylabel, series in plots:
        figure, axis = plt.subplots()
        for label, values in series:
            axis.plot(epochs, values, label=label)
        axis.set(title=title, xlabel="Epoch", ylabel=ylabel)
        if len(series) > 1:
            axis.legend()
        figure.tight_layout()
        figure.savefig(WORKING_DIR / filename)
        plt.close(figure)


def run_training():
    set_seed(CONFIG["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_records, val_records = split_patients(pd.read_csv(TRAIN_CSV))
    print(f"Device: {device}")
    for label, frame in (("Training", train_records), ("Validation", val_records)):
        print(f"{label}: {len(frame)} images, {frame['PatientID'].nunique()} patients")
        print(frame["Diagnosis"].value_counts())

    # Export the split for inspection, but pass its records directly to datasets.
    train_records.to_csv(WORKING_DIR / "train_split_p.csv", index=False)
    val_records.to_csv(WORKING_DIR / "val_split_p.csv", index=False)
    training = CancerDataset(BF_TRAIN_DIR, FL_TRAIN_DIR, train_records, PairedImageTransform(training=True))
    validation = CancerDataset(BF_TRAIN_DIR, FL_TRAIN_DIR, val_records)
    train_loader = make_loader(training, device, shuffle=True)
    val_loader = make_loader(validation, device)

    model = CancerNet().to(device)
    # Balance positive examples using class counts from the training split only.
    positives = train_records["Diagnosis"].eq(1).sum()
    negatives = train_records["Diagnosis"].eq(0).sum()
    if positives == 0 or negatives == 0:
        raise ValueError("The training split must contain both diagnosis classes.")
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([negatives / positives], device=device))
    optimizer = torch.optim.AdamW([
        {"params": model.model.features.parameters()},
        {"params": model.model.classifier.parameters()},
    ], lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=CONFIG["lr_factor"], patience=CONFIG["lr_patience"],
    )

    history = {"loss": [], "train_auc": [], "val_auc": []}
    best_auc, stale_epochs = -float("inf"), 0
    for epoch in range(1, CONFIG["epochs"] + 1):
        loss = train_epoch(model, train_loader, optimizer, criterion, device)
        # Training AUC uses the augmented loader; validation uses unmodified images.
        train_auc = evaluate_auc(model, train_loader, device)
        val_auc = evaluate_auc(model, val_loader, device)
        # Validation AUC controls both the plateau scheduler and checkpoint selection.
        scheduler.step(val_auc)
        for key, value in (("loss", loss), ("train_auc", train_auc), ("val_auc", val_auc)):
            history[key].append(value)
        print(f"Epoch {epoch}/{CONFIG['epochs']} | Loss {loss:.4f} | Train AUC {train_auc:.4f} | Val AUC {val_auc:.4f}")
        print(f"Learning rates: {scheduler.get_last_lr()}")

        # Always save the first epoch, then replace it only when validation improves.
        if val_auc > best_auc:
            best_auc, stale_epochs = val_auc, 0
            torch.save(model.state_dict(), WORKING_DIR / "best_model_p.pth")
            print(f"Checkpoint saved: validation AUC {best_auc:.4f}")
        else:
            stale_epochs += 1
        if stale_epochs >= CONFIG["early_stopping_patience"]:
            print(f"Training stopped after {stale_epochs} epochs without improvement.")
            break

    save_learning_curves(history)
    return history


In [ ]:
history = run_training()


## Test predictions and CSV export


In [ ]:
@torch.inference_mode()
def predict_test():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = WORKING_DIR / "best_model_p.pth"
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Train the model before prediction: {checkpoint}")

    dataset = CancerDataset(BF_TEST_DIR, FL_TEST_DIR)
    loader = make_loader(dataset, device)
    # The checkpoint supplies all parameters; no pretrained download is needed here.
    model = CancerNet(weights=None).to(device)
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    model.eval()
    print(f"Predicting {len(dataset)} image pairs on {device}")

    filenames, probabilities = [], []
    for images, names in loader:
        logits = model(images.to(device, non_blocking=True))
        filenames.extend(names)
        probabilities.append(logits.sigmoid().flatten())

    # Transfer probabilities once and retain deterministic filename ordering.
    submission = pd.DataFrame({
        "Name": filenames,
        "Diagnosis": torch.cat(probabilities).cpu().numpy().astype(float),
    })
    destination = WORKING_DIR / "sampleSubmission_p.csv"
    submission.to_csv(destination, index=False)
    print(f"Predictions written to {destination}")
    return submission


In [ ]:
submission = predict_test()
